In [1]:
# GPU 사용 가능 여부 확인
import sys
import torch
import random
import numpy as np

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Python version: 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
PyTorch version: 2.6.0+cu118
CUDA version: 11.8
GPU available: True
GPU name: NVIDIA GeForce RTX 4090


In [2]:
# 모델 재현을 위한 랜덤시드 고정
def set_random_seed(seed_value=42):
    # Python의 기본 난수 시드 설정
    random.seed(seed_value)
    # NumPy 난수 시드 설정
    np.random.seed(seed_value)
    # PyTorch 난수 시드 설정 (CPU)
    torch.manual_seed(seed_value)
    # PyTorch 난수 시드 설정 (GPU)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    # CuDNN 설정
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed()

# matplotlib default 설정
import matplotlib.pyplot as plt
plt.style.use('default')

In [3]:
# 사전학습된 가중치 파일 불러오기
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

In [4]:
# train 파라미터 설정
train_params = {
    'data': '/home/leedh/바탕화면/MCT_for_ChamDog/model_training/dataset_yolo_split/data.yaml', 
    'epochs': 200,
    'imgsz': 640,
    'batch': 16,
    'project': './runs',
    'name': 'v11n',
    'seed': 42,
    'optimizer': 'AdamW',           # default='auto' - Options include SGD, Adam, AdamW, NAdam, RAdam, RMSProp etc.
    'lr0': 0.0001,
    'lrf': 0.01,
    'conf': 0.10,                   # default: 0.25
    'save_json': False,
    'workers': 4,                   # default: 8
    'patience': 20,
}

In [5]:
# 모델 학습
results = model.train(**train_params)

New https://pypi.org/project/ultralytics/8.3.231 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.216 🚀 Python-3.10.18 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24058MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.1, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/leedh/바탕화면/MCT_for_ChamDog/model_training/dataset_yolo_split/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=

# Validation set

In [6]:
from ultralytics import YOLO
best_model = YOLO(f"{train_params['project']}/{train_params['name']}/weights/best.pt")

# validation 파라미터 설정
val_params = {
    'data': train_params['data'],
    'batch': train_params['batch'],
    'conf': train_params['conf'],
    'save_json': True,
    'project': f"{train_params['project']}/{train_params['name']}",
    'name': 'val_performance',
    'split': 'val',                        # 'val', 'test', 'train'
    'workers': 0,
}

In [7]:
# 모델 학습
set_random_seed()
val_results = best_model.val(**val_params)

Ultralytics 8.3.216 🚀 Python-3.10.18 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24058MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7565.9±2906.3 MB/s, size: 12074.1 KB)
val: Scanning /home/leedh/바탕화면/MCT_for_ChamDog/model_training/dataset_yolo_split/labels/val.cache... 59 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 59/59 182.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 0.4it/s 10.8s.4ss
                   all         59        147      0.927      0.932      0.975      0.915
Speed: 0.1ms preprocess, 1.8ms inference, 0.0ms loss, 0.2ms postprocess per image
Saving /home/leedh/바탕화면/MCT_for_ChamDog/model_training/runs/v11n/val_performance/predictions.json...
Results saved to /home/leedh/바탕화면/MCT_for_ChamDog/model_training/runs/v11n/val_performance


In [ ]:
# 실제 validation에 대한 predict 결과 저장
pred_results = best_model(
    source='./dataset_yolo_split/images/val',
    conf=train_params['conf'],          # default: 0.25
    save=True,
    project=f"{train_params['project']}/{train_params['name']}",    # Name of the project directory where training outputs are saved
    name='val_predict',                 # Name of the project
)

In [ ]:
# 실제 validation에 대한 predict 결과 저장
pred_results = best_model(
    source='./dataset_yolo_split/images/test',
    conf=train_params['conf'],          # default: 0.25
    save=True,
    project=f"{train_params['project']}/{train_params['name']}",    # Name of the project directory where training outputs are saved
    name='test_predict',                 # Name of the project
)